# Training times per approach

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('.').resolve()))  # this notebook's own dir (analysis/), for local imports

from training_times import ROOT, REAL_TRIM_NAMES, real_logs, APPROACH_ORDER, build_timing_table, _fmt_duration

timing_df = build_timing_table()
timing_df['approach'] = pd.Categorical(timing_df['approach'], categories=APPROACH_ORDER, ordered=True)
timing_df = timing_df.sort_values(['approach', 'dataset', 'trim'])

print(f'Datasets: {len(real_logs)}   Trims: {len(REAL_TRIM_NAMES)}   '
      f'Approaches: {timing_df["approach"].nunique()} ({APPROACH_ORDER})')


## Event-log-based approaches (Amiri, Bukhsh, Camargo)

In [ ]:
event_log_df = timing_df[timing_df['approach'].isin(['amiri', 'bukhsh', 'camargo'])]
print(event_log_df[['dataset','trim','approach','regime','single_run','single_run_kind',
                     'full_space','full_space_kind','source']].to_string(index=False))


## Time-series baselines (incl. Chronos, TabPFN)

In [ ]:
ts_df = timing_df[timing_df['approach'].astype(str).str.startswith('ts_')]
print(ts_df[['dataset','trim','approach','regime','series','single_run','full_space']]
      .to_string(index=False))


## Coverage and summary

In [ ]:
found   = timing_df.dropna(subset=['single_run_s'])
missing = timing_df[timing_df['single_run_s'].isna()]

print(f'Found  : {len(found)}/{len(timing_df)} rows')
print(f'Missing: {len(missing)}/{len(timing_df)} rows')
if len(missing):
    print()
    print('Missing rows:')
    print(missing[['dataset','trim','approach','series']].to_string(index=False))

print()
print('Per-approach summary — single_run_s (seconds):')
summary = found.groupby('approach', observed=True)['single_run_s'].agg(['count','mean','min','max']).round(1)
summary.insert(0, 'regime', [('inference' if a in ('ts_chronos', 'ts_tabpfn') else 'training')
                              for a in summary.index])
print(summary.to_string())

print()
print('single_run_kind / full_space_kind exact-vs-approx breakdown:')
print(found.groupby('approach', observed=True)[['single_run_kind','full_space_kind']]
      .apply(lambda g: pd.Series({
          'single_run_approx_pct': round((g['single_run_kind'] == 'approx').mean() * 100, 1),
          'full_space_approx_pct': round((g['full_space_kind'] == 'approx').mean() * 100, 1),
      })).to_string())


## Averaged summary tables

`ts_*` approaches have a `series` dimension (concurrent_cases / throughput_time) that
amiri/bukhsh/camargo don't — both tables below average across series too, so every
approach ends up with exactly one number per trim/dataset cell.

### Average `single_run_s` by trim (mean across datasets)

In [ ]:
from training_times import build_avg_by_trim, build_avg_by_dataset, _fmt_table

by_trim_single = build_avg_by_trim(timing_df, 'single_run_s')
print(_fmt_table(by_trim_single).to_string())

### Average `single_run_s` by dataset (mean across trims)

In [ ]:
by_dataset_single = build_avg_by_dataset(timing_df, 'single_run_s')
print(_fmt_table(by_dataset_single).to_string())

### Overall average training time — one row per Excel export

Single row, one column per approach: mean `single_run_s` across every dataset/trim(/series).
`ts_*` columns (all sktime baselines, Chronos, TabPFN) are blanked to `-` — none of them
have a meaningful *training* time to report (baselines: near-instant/not HPO-driven;
Chronos/TabPFN: inference-only, see the `regime` column above). Only amiri/bukhsh/camargo
get a real number.

In [ ]:
from training_times import build_avg_training_time_row, save_avg_training_time_excel


avg_row = build_avg_training_time_row(timing_df, 'single_run_s')
print(avg_row.to_string(index=False))

xlsx_path = save_avg_training_time_excel(timing_df, value_col='single_run_s')
print(f'\nSaved -> {xlsx_path.relative_to(ROOT)}')